# Customer Churn Prediction System
### Phase 3: Prepare for Machine Learning (Tasks 8-10)

Phase 2 told us `Payment Delay` and `Support Calls` are the strongest predictors,
`CustomerID` is already dropped, and the data has no missing values or duplicates.
Now we get the data into the exact shape a model can actually train on.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('customer_churn.csv')
df = df.drop(columns=['CustomerID'])  # same cleaning step as Phase 2
print(f"Working dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Working dataset: 64374 rows, 11 columns


## Task 8: Split Features (X) from Target (y)

- **`X`** = every column except `Churn` — what the model looks at.
- **`y`** = only the `Churn` column — what the model is trying to predict.

Think of `X` as the question paper and `y` as the answer key.

In [2]:
X = df.drop(columns=['Churn'])
y = df['Churn']

print(f"X shape (features): {X.shape}")
print(f"y shape (target):   {y.shape}")
print(f"\nColumns in X: {list(X.columns)}")

X shape (features): (64374, 10)
y shape (target):   (64374,)

Columns in X: ['Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction']


## Task 9: Train-Test Split (80/20)

We hold back 20% of the data that the model never sees during training, so we can
honestly check afterward how well it actually learned — not how well it memorized.

`stratify=y` forces both the training set and the testing set to keep the same
churn ratio as the full dataset (~53% stayed / ~47% churned), so neither set ends
up accidentally skewed. `random_state=42` just makes the split reproducible — same
split every time this cell runs.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} customers")
print(f"Testing set:  {X_test.shape[0]} customers")

Training set: 51499 customers
Testing set:  12875 customers


In [4]:
# Confirm stratify worked -- churn ratio should be nearly identical across all three
print("Churn rate in full dataset:", round(y.mean(), 4))
print("Churn rate in training set:", round(y_train.mean(), 4))
print("Churn rate in testing set: ", round(y_test.mean(), 4))

Churn rate in full dataset: 0.4737
Churn rate in training set: 0.4737
Churn rate in testing set:  0.4737


**Result:** all three churn rates should land within a hair of each other
(~0.47). That confirms the split is representative — the model will train on a
mini-version of the same population it'll be tested on, not some skewed subset.

## Task 10: Encoding + Scaling

Models only understand numbers, so text categories need converting first. We do
this **after** splitting, and fit anything that learns from the data's statistics
(the scaler) only on the training set — otherwise information from the "hidden"
test set would leak into training and make our later evaluation dishonest.

We'll work on copies of `X_train` / `X_test` so the original stays untouched.

In [6]:
X_train_prep = X_train.copy()
X_test_prep = X_test.copy()

# --- Step 1: Encode Gender (binary, so a simple 0/1 map is enough) ---
gender_map = {'Male': 0, 'Female': 1}
X_train_prep['Gender'] = X_train_prep['Gender'].map(gender_map)
X_test_prep['Gender'] = X_test_prep['Gender'].map(gender_map)

print("Gender encoded. Sample:")
X_train_prep[['Gender']].head()

Gender encoded. Sample:


,Gender
17402,1
20461,1
63787,0
12218,0
6831,1


In [7]:
# --- Step 2: One-hot encode Subscription Type and Contract Length ---
# These are nominal categories (no real order), so we one-hot encode instead of
# assigning arbitrary numbers -- that avoids implying a fake ranking like
# Premium > Standard > Basic in the model's eyes.
categorical_cols = ['Subscription Type', 'Contract Length']

X_train_prep = pd.get_dummies(X_train_prep, columns=categorical_cols, drop_first=True)
X_test_prep = pd.get_dummies(X_test_prep, columns=categorical_cols, drop_first=True)

# Make sure both sets end up with the exact same columns in the same order
X_test_prep = X_test_prep.reindex(columns=X_train_prep.columns, fill_value=0)

print(f"Columns after encoding: {list(X_train_prep.columns)}")
print(f"\nX_train shape: {X_train_prep.shape}   X_test shape: {X_test_prep.shape}")

Columns after encoding: ['Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Total Spend', 'Last Interaction', 'Subscription Type_Premium', 'Subscription Type_Standard', 'Contract Length_Monthly', 'Contract Length_Quarterly']

X_train shape: (51499, 12)   X_test shape: (12875, 12)


In [8]:
# --- Step 3: Scale the numeric columns ---
# Fit the scaler on TRAINING data only, then apply that same scaling to both sets.
numeric_cols = ['Age', 'Tenure', 'Usage Frequency', 'Support Calls',
                'Payment Delay', 'Total Spend', 'Last Interaction']

scaler = StandardScaler()
X_train_prep[numeric_cols] = scaler.fit_transform(X_train_prep[numeric_cols])
X_test_prep[numeric_cols] = scaler.transform(X_test_prep[numeric_cols])

print("Numeric columns scaled. Preview:")
X_train_prep.head()

Numeric columns scaled. Preview:


,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Subscription Type_Premium,Subscription Type_Standard,Contract Length_Monthly,Contract Length_Quarterly
17402,1.002290,1,0.467698,-0.347864,-1.413220,-1.935572,0.728167,0.984201,False,True,True,False
20461,-0.289317,1,-0.117619,-1.254923,-0.448905,-0.918315,-1.387161,-1.100821,False,True,False,False
63787,-1.437413,0,-0.117619,0.672577,0.836848,0.211971,0.670685,-1.564159,True,False,False,True
12218,0.212974,0,-1.171191,1.239489,1.479724,-0.805286,1.759006,1.563374,True,False,True,False
6831,1.217558,1,0.292103,-0.234481,0.193971,0.551057,0.712838,-1.100821,False,True,False,True


In [9]:
# Final check -- everything should now be numeric, no text columns left
print(X_train_prep.dtypes)
print(f"\nAny non-numeric columns left? {(X_train_prep.dtypes == 'object').any()}")

Age                           float64
Gender                          int64
Tenure                        float64
Usage Frequency               float64
Support Calls                 float64
Payment Delay                 float64
Total Spend                   float64
Last Interaction              float64
Subscription Type_Premium        bool
Subscription Type_Standard       bool
Contract Length_Monthly          bool
Contract Length_Quarterly        bool
dtype: object

Any non-numeric columns left? False


**Why `fit_transform` on train but only `transform` on test?**
- `fit_transform` on `X_train` calculates the mean and standard deviation *from the
  training data* and applies scaling using those numbers.
- `transform` on `X_test` reuses those *same* training-derived numbers — it does
  not recalculate anything from the test set.
- If we let the test set influence the scaling calculation, the model would
  indirectly get a sneak peek at test data during "training," which would make
  Phase 5's evaluation results artificially better than they'd be in the real
  world. This is the single most important rule in this whole phase.

---
### ✅ Phase 3 checkpoint

You now have four ready-to-use pieces sitting in memory:
- `X_train_prep`, `y_train` — fully numeric, scaled, encoded training data
- `X_test_prep`, `y_test` — the untouched-until-now testing data, prepped the same way

Every column is numeric, every categorical variable is properly encoded, every
numeric column is scaled, and there's zero leakage from the test set into training.

**Next: Phase 4 (Task 11)** — feeding `X_train_prep` / `y_train` into four different
models (Logistic Regression, Random Forest, SVM, KNN) and generating predictions on
`X_test_prep`.